# Pipeline Orchestration (Optional)

This notebook provides a practical orchestration plan for scheduling the full pipeline:
1. ELT process
2. Data quality checks

It includes templates for:
- Cron jobs
- GitHub Actions (CI/CD schedule)
- Airflow DAG (framework option)


## 1) Objective

Automate recurring runs of:
- `main.ipynb` (ELT pipeline)
- `data_quality_test_plan.ipynb` (DQ validation)

Run frequency example:
- Daily at 02:00 (local time)
- Plus manual trigger support


In [2]:
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "m2-project").exists():
    PROJECT_ROOT = PROJECT_ROOT / "m2-project"

ELT_NOTEBOOK = PROJECT_ROOT / "main.ipynb"
DQ_NOTEBOOK = PROJECT_ROOT / "data_quality_test_plan.ipynb"

print("Project root:", PROJECT_ROOT)
print("ELT notebook exists:", ELT_NOTEBOOK.exists(), ELT_NOTEBOOK)
print("DQ notebook exists:", DQ_NOTEBOOK.exists(), DQ_NOTEBOOK)
print("Generated at:", datetime.now().isoformat())

Project root: /home/franklin/dsai/dsai-5m-projects/m2-project
ELT notebook exists: True /home/franklin/dsai/dsai-5m-projects/m2-project/main.ipynb
DQ notebook exists: True /home/franklin/dsai/dsai-5m-projects/m2-project/data_quality_test_plan.ipynb
Generated at: 2026-03-27T20:01:27.954992


## 2) Pipeline Design

Recommended task order:
1. Run ELT notebook
2. Run DQ notebook
3. Save logs and fail pipeline when DQ fails

Dependency:
- DQ depends on ELT completion


In [3]:
ELT_EXEC_CMD = "jupyter nbconvert --to notebook --inplace --execute main.ipynb"
DQ_EXEC_CMD = "jupyter nbconvert --to notebook --inplace --execute data_quality_test_plan.ipynb"

print("ELT command:", ELT_EXEC_CMD)
print("DQ command:", DQ_EXEC_CMD)

ELT command: jupyter nbconvert --to notebook --inplace --execute main.ipynb
DQ command: jupyter nbconvert --to notebook --inplace --execute data_quality_test_plan.ipynb


## 3) Option A: Cron Job

Use cron if you want lightweight scheduling on one machine.


In [4]:
cron_script = """#!/usr/bin/env bash
set -euo pipefail

cd /home/franklin/dsai/dsai-5m-projects/m2-project

# Activate conda environment (adjust as needed)
source ~/miniconda3/etc/profile.d/conda.sh
conda activate elt

# Run ELT then DQ
jupyter nbconvert --to notebook --inplace --execute main.ipynb
jupyter nbconvert --to notebook --inplace --execute data_quality_test_plan.ipynb
"""

cron_entry = "0 2 * * * /home/franklin/dsai/dsai-5m-projects/m2-project/scripts/run_pipeline.sh >> /home/franklin/dsai/dsai-5m-projects/m2-project/logs/pipeline.log 2>&1"

print("Create script: scripts/run_pipeline.sh")
print(cron_script)
print("")
print("Add this to crontab -e:")
print(cron_entry)


Create script: scripts/run_pipeline.sh
#!/usr/bin/env bash
set -euo pipefail

cd /home/franklin/dsai/dsai-5m-projects/m2-project

# Activate conda environment (adjust as needed)
source ~/miniconda3/etc/profile.d/conda.sh
conda activate elt

# Run ELT then DQ
jupyter nbconvert --to notebook --inplace --execute main.ipynb
jupyter nbconvert --to notebook --inplace --execute data_quality_test_plan.ipynb


Add this to crontab -e:
0 2 * * * /home/franklin/dsai/dsai-5m-projects/m2-project/scripts/run_pipeline.sh >> /home/franklin/dsai/dsai-5m-projects/m2-project/logs/pipeline.log 2>&1


## 4) Option B: GitHub Actions (CI/CD Scheduled)

Good default for collaboration, visibility, and alerting.

This workflow runs on:
- Daily schedule (`cron`)
- Manual trigger (`workflow_dispatch`)


In [5]:
github_actions_yaml = """name: Scheduled ELT + DQ

on:
  schedule:
    - cron: '0 18 * * *'   # 02:00 Asia/Singapore (UTC+8)
  workflow_dispatch:

jobs:
  run-pipeline:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install jupyter duckdb pandas great-expectations

      - name: Run ELT notebook
        working-directory: m2-project
        run: jupyter nbconvert --to notebook --inplace --execute main.ipynb

      - name: Run DQ notebook
        working-directory: m2-project
        run: jupyter nbconvert --to notebook --inplace --execute data_quality_test_plan.ipynb
"""

print("Create file: .github/workflows/pipeline-orchestration.yml")
print(github_actions_yaml)

Create file: .github/workflows/pipeline-orchestration.yml
name: Scheduled ELT + DQ

on:
  schedule:
    - cron: '0 18 * * *'   # 02:00 Asia/Singapore (UTC+8)
  workflow_dispatch:

jobs:
  run-pipeline:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install jupyter duckdb pandas great-expectations

      - name: Run ELT notebook
        working-directory: m2-project
        run: jupyter nbconvert --to notebook --inplace --execute main.ipynb

      - name: Run DQ notebook
        working-directory: m2-project
        run: jupyter nbconvert --to notebook --inplace --execute data_quality_test_plan.ipynb



## 5) Option C: Airflow DAG

Use this if you need richer dependency management and operational UI.


In [6]:
airflow_dag_py = """from datetime import datetime
from airflow import DAG
from airflow.operators.bash import BashOperator

with DAG(
    dag_id='olist_elt_dq_pipeline',
    start_date=datetime(2026, 1, 1),
    schedule='0 2 * * *',
    catchup=False,
    tags=['olist', 'elt', 'dq'],
) as dag:

    run_elt = BashOperator(
        task_id='run_elt_notebook',
        bash_command='cd /opt/airflow/dags/m2-project && jupyter nbconvert --to notebook --inplace --execute main.ipynb',
    )

    run_dq = BashOperator(
        task_id='run_dq_notebook',
        bash_command='cd /opt/airflow/dags/m2-project && jupyter nbconvert --to notebook --inplace --execute data_quality_test_plan.ipynb',
    )

    run_elt >> run_dq
"""

print("Create file: dags/olist_elt_dq_pipeline.py")
print(airflow_dag_py)

Create file: dags/olist_elt_dq_pipeline.py
from datetime import datetime
from airflow import DAG
from airflow.operators.bash import BashOperator

with DAG(
    dag_id='olist_elt_dq_pipeline',
    start_date=datetime(2026, 1, 1),
    schedule='0 2 * * *',
    catchup=False,
    tags=['olist', 'elt', 'dq'],
) as dag:

    run_elt = BashOperator(
        task_id='run_elt_notebook',
        bash_command='cd /opt/airflow/dags/m2-project && jupyter nbconvert --to notebook --inplace --execute main.ipynb',
    )

    run_dq = BashOperator(
        task_id='run_dq_notebook',
        bash_command='cd /opt/airflow/dags/m2-project && jupyter nbconvert --to notebook --inplace --execute data_quality_test_plan.ipynb',
    )

    run_elt >> run_dq



## 6) Recommended Choice for This Project

Use **GitHub Actions** first because it is simple, auditable, and team-friendly.

Fallback option:
- Use **cron** for local-only execution if CI secrets/network constraints are blocking cloud runs.


## 7) Operational Checklist

1. Ensure runtime dependencies are installed (`jupyter`, `duckdb`, `pandas`, `great-expectations`).
2. Verify ELT notebook executes without manual steps.
3. Verify DQ notebook returns clear pass/fail outcomes.
4. Add scheduled trigger (cron or CI/CD).
5. Store logs/artifacts and configure failure notifications.
